# S4 · 종단 패널과 코호트

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [S4 · 종단 패널과 코호트](https://grow.minds.kr/textbooks/css-methods/causal/book/s4-시나리오-종단-패널과-코호트.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 이탈은 무작위가 아니다
2차에 안 온 80명은 1차부터 달랐다. *d* = 0.86 은 이 책에서 가장 큰 차이다.

In [ ]:
pan = load("panel")
w1 = pan[pan.wave == 1].set_index("id"); w2 = pan[pan.wave == 2].set_index("id")
stay = w1.index.intersection(w2.index); left = w1.index.difference(w2.index)
print(len(stay), len(left),
      round(w1.loc[stay].mil.mean(), 2), round(w1.loc[left].mil.mean(), 2),
      round(cohen_d(w1.loc[stay].mil, w1.loc[left].mil), 2))   # 420 80 5.01 4.15 0.86

## 3. 개인 안에서 보면 관계가 줄어든다
횡단 .286 이 개인 평균 중심화 후 .056 으로. 대부분이 **안정 성향의 몫**이었다.

In [ ]:
p2 = pan.copy()
for cc in ("hjs", "mil"):
    p2[cc + "_c"] = p2[cc] - p2.groupby("id")[cc].transform("mean")
print(round(float(w1.hjs.corr(w1.mil)), 3), round(float(p2.hjs_c.corr(p2.mil_c)), 3))

## 4. 교차지연: 두 방향을 나란히
앞 시점의 자기 자신을 통제하고 상대를 넣는다. 비대칭이 방향의 힌트다(증명은 아니다).

In [ ]:
wide = pan.pivot(index="id", columns="wave", values=["hjs", "mil"]).dropna()
z = lambda v: (v - v.mean()) / v.std(ddof=1)
for lab, (y, y1, x1) in {"여정→의미 1→2": (("mil",2), ("mil",1), ("hjs",1)),
                          "여정→의미 2→3": (("mil",3), ("mil",2), ("hjs",2)),
                          "의미→여정 1→2": (("hjs",2), ("hjs",1), ("mil",1)),
                          "의미→여정 2→3": (("hjs",3), ("hjs",2), ("mil",2))}.items():
    b, se, p, _ = ols(z(wide[y]), [z(wide[y1]), z(wide[x1])])
    print(lab, round(b[2], 3), round(float(p[2]), 3))

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.